# Integrated Machine Learning Systems Project
## Hybrid Reinforcement Learning, Ensemble Learning, LLM & RAG System

**Author:** Nobuhle Ngobese  
**Student Number:** 202413385  
**Module:** Machine Learning  

---

This notebook implements two advanced intelligent systems using South African datasets:

- **Component A:** Hybrid Reinforcement Learning and Ensemble Learning System (Traffic Accidents)
- **Component B:** LLM and Retrieval-Augmented Generation System (Parliamentary Hansard Transcripts)

---

## Setup and Dependencies

In [ ]:
# Install required packages into Jupyter's Python environment
import sys
!{sys.executable} -m pip install xgboost sentence-transformers transformers faiss-cpu --quiet

In [ ]:
import sys, os

# Ensure the notebook can find component_a.py and component_b.py
PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'Project directory: {PROJECT_DIR}')
print('All base imports successful.')

---
# COMPONENT A
## Hybrid Reinforcement Learning and Ensemble Learning System

Using the South African Traffic Accident Dataset, we design a hybrid intelligent system that:
1. Preprocesses accident data
2. Trains ensemble models for severity prediction
3. Uses Q-learning for adaptive intervention strategies
4. Integrates predictions with learned policies

---

## A1. Data Preparation and Preprocessing

**Preprocessing decisions:**
- **Missing values:** Numeric columns imputed with median (robust to outliers), categorical with mode (preserves distribution).
- **Categorical encoding:** Label encoding applied to convert text categories to numeric features for tree-based models.
- **Outlier treatment:** IQR-based clipping used to reduce the influence of extreme values without removing data points.

These choices are justified because tree-based ensemble models (Random Forest, XGBoost) handle encoded categoricals well, and median imputation is preferred over mean when outliers are present.

In [ ]:
from component_a import load_and_prepare_data, preprocess

# Pass your CSV path here if you have the dataset, otherwise synthetic data is generated
# df_accidents = load_and_prepare_data('data/car_accidents.csv')
df_accidents = load_and_prepare_data()

In [ ]:
# Explore raw data
print('First 5 rows:')
display(df_accidents.head())
print(f'\nShape: {df_accidents.shape}')
print(f'\nData types:\n{df_accidents.dtypes}')
print(f'\nDescriptive statistics:')
display(df_accidents.describe())

In [ ]:
# Visualise target distribution before preprocessing
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_accidents['Severity'].value_counts().plot(kind='bar', ax=axes[0], color=['green','orange','red'])
axes[0].set_title('Severity Distribution')
axes[0].set_ylabel('Count')

df_accidents['Province'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Accidents by Province')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Preprocess
df_processed, label_encoders = preprocess(df_accidents)
df_processed.to_csv('data/preprocessed_accidents.csv', index=False)
print('\nPreprocessed data saved.')

In [ ]:
# Correlation heatmap after encoding
plt.figure(figsize=(10, 8))
sns.heatmap(df_processed.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## A2. Ensemble Learning for Accident Prediction

We train two ensemble models:
- **Random Forest** — bagging-based, reduces variance
- **XGBoost** — boosting-based, reduces bias

Both are evaluated using Accuracy, Precision, Recall, F1-score, and confusion matrices.

In [ ]:
from component_a import train_ensemble_models

ensemble_results, X_test, y_test = train_ensemble_models(df_processed)

In [ ]:
# Summary comparison table
comparison = pd.DataFrame({
    name: {k: v for k, v in res.items() if k in ['Accuracy','Precision','Recall','F1']}
    for name, res in ensemble_results.items()
}).T
display(comparison.style.format('{:.4f}').background_gradient(cmap='Greens'))

### Bias-Variance Trade-off Discussion

| Aspect | Random Forest | XGBoost |
|--------|--------------|--------|
| Strategy | Bagging (parallel trees) | Boosting (sequential trees) |
| Bias | Moderate — limited by max_depth | Low — corrects errors iteratively |
| Variance | Low — averaging reduces variance | Higher — can overfit without regularization |
| Overfitting risk | Lower | Higher (mitigated by learning_rate, max_depth) |
| Interpretability | Feature importance available | Feature importance + SHAP available |

Random Forest is more robust to noisy data, while XGBoost typically achieves higher accuracy on structured data when properly tuned. The trade-off is stability vs. performance.

## A3. Reinforcement Learning for Accident Prevention

We model accident prevention as a **Markov Decision Process (MDP)**:

| MDP Element | Definition |
|-------------|------------|
| **States** | (Road Condition, Accident Frequency Level) — 4 × 3 = 12 states |
| **Actions** | No Action, Safety Campaign, Increase Enforcement, Infrastructure Upgrade |
| **Rewards** | Positive for reducing severity; negative for inaction during high-risk states |
| **Transitions** | Stochastic — actions probabilistically reduce frequency; road conditions change randomly |

A **Q-learning** agent learns the optimal intervention policy through trial-and-error interaction with the environment.

In [ ]:
from component_a import train_q_learning

Q_table, mdp, rewards_history = train_q_learning(episodes=2000)

In [ ]:
# Display Q-table as heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(Q_table, annot=True, fmt='.1f', cmap='YlGnBu',
            xticklabels=mdp.actions,
            yticklabels=[mdp.state_name(s) for s in range(mdp.n_states)])
plt.title('Q-Table: State-Action Values')
plt.xlabel('Action')
plt.ylabel('State')
plt.tight_layout()
plt.show()

### Convergence Analysis

The reward plot above shows:
- **Early episodes:** High variance due to exploration (high epsilon). The agent tries random actions.
- **Later episodes:** Rewards stabilise as the agent exploits learned Q-values.
- **Smoothed curve (red):** Shows clear upward trend, confirming policy improvement over time.

The agent converges to a stable policy where it recommends infrastructure upgrades for hazardous road conditions and enforcement for high-frequency accident areas.

## A4. System Integration and Evaluation

We integrate the ensemble model's severity predictions with the RL agent's intervention policy. The ensemble model identifies high-risk scenarios, and the RL agent recommends the optimal intervention.

In [ ]:
from component_a import integrate_system

integrate_system(ensemble_results, Q_table, mdp, df_processed)

### Exploration-Exploitation Trade-off

The epsilon-greedy strategy decays from 1.0 (full exploration) to 0.01 (near-full exploitation):
- **Exploration** discovers that infrastructure upgrades yield highest rewards on icy/gravel roads.
- **Exploitation** ensures the agent consistently recommends the best-known action.
- Without sufficient exploration, the agent might never discover that infrastructure upgrades outperform enforcement on gravel roads.

### Ethical Implications
- Automated intervention recommendations must be validated by domain experts before deployment.
- Under-reported rural accidents may bias the model toward urban-focused interventions.
- The system should be used as **decision-support**, not autonomous decision-making.

---

---
# COMPONENT B
## Large Language Model and Retrieval-Augmented Generation System

Using South African Parliamentary Hansard Transcripts, we build:
1. Text preprocessing and embedding pipeline
2. Sentiment classifier (fine-tuned transformer or equivalent)
3. RAG pipeline for policy-focused question answering
4. Ethics and responsible AI discussion

---

## B1. Data Preparation and Embedding Generation

**Preprocessing decisions:**
- **Cleaning:** Lowercasing, removal of special characters, whitespace normalisation.
- **Tokenization:** BPE (Byte-Pair Encoding) via BERT tokenizer — handles out-of-vocabulary words common in South African English (Afrikaans/Zulu terms).
- **Embeddings:** Sentence-transformers (all-MiniLM-L6-v2) produce 384-dimensional dense vectors capturing semantic meaning. TF-IDF used as fallback if transformers unavailable.

BPE is preferred over word-level tokenization because parliamentary transcripts contain domain-specific terminology that may not appear in standard vocabularies.

In [ ]:
from component_b import load_hansard_data, clean_and_tokenize, generate_embeddings

# Pass your CSV path here if you have the dataset, otherwise synthetic data is generated
# df_hansard = load_hansard_data('data/hansard_transcripts.csv')
df_hansard = load_hansard_data()

In [ ]:
# Explore raw data
display(df_hansard.head())
print(f'\nSentiment distribution:\n{df_hansard["sentiment"].value_counts()}')
print(f'\nTopics:\n{df_hansard["topic"].value_counts()}')

In [ ]:
# Visualise data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_hansard['sentiment'].value_counts().plot(kind='bar', ax=axes[0], color=['green','red','gray'])
axes[0].set_title('Sentiment Distribution')
df_hansard['topic'].value_counts().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Topic Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Clean and tokenize
df_hansard = clean_and_tokenize(df_hansard)

In [ ]:
# Generate embeddings
embeddings = generate_embeddings(df_hansard)

# Save preprocessed data
df_hansard[['text','clean_text','topic','year','sentiment']].to_csv(
    'data/preprocessed_hansard.csv', index=False)
print('Preprocessed Hansard data saved.')

## B2. Fine-Tuning a Transformer Model

We fine-tune a pre-trained model for **policy sentiment classification** (positive / negative / neutral).

- Primary approach: BERT fine-tuning using HuggingFace Transformers
- Fallback: Gradient Boosting on transformer embeddings (if GPU/memory unavailable)

Both approaches leverage the semantic representations from the embedding step.

In [ ]:
from component_b import train_sentiment_classifier

sentiment_results = train_sentiment_classifier(df_hansard, embeddings)

In [ ]:
# Display results
results_df = pd.DataFrame([sentiment_results], index=['Sentiment Classifier'])
display(results_df.style.format('{:.4f}').background_gradient(cmap='Greens'))

### Confusion Matrix Interpretation

The confusion matrix above shows:
- **Diagonal values:** Correct predictions per class
- **Off-diagonal values:** Misclassifications
- Neutral sentiment is often hardest to classify as it overlaps with both positive and negative language
- The model performs best on negative sentiment due to stronger linguistic markers (e.g., "failure", "condemn", "unacceptable")

## B3. Retrieval-Augmented Generation (RAG) Implementation

The RAG pipeline:
1. **Retrieval:** FAISS vector index finds the top-k most semantically similar documents to a query
2. **Generation:** A language model generates an answer grounded in the retrieved context

This approach reduces hallucination by constraining the model to answer based on actual parliamentary records.

In [ ]:
from component_b import build_rag_pipeline

retrieve_fn, generate_fn = build_rag_pipeline(df_hansard, embeddings)

In [ ]:
# Additional custom query demonstration
custom_query = "What is the parliamentary view on healthcare budget allocation?"
print(f"Query: {custom_query}\n")
retrieved_docs = retrieve_fn(custom_query)
print("Retrieved documents:")
for i, row in retrieved_docs.iterrows():
    print(f"  [{row['sentiment']}] {row['text'][:100]}...")
print(f"\nGenerated answer:")
print(generate_fn(custom_query, retrieved_docs))

### RAG Evaluation

| Criterion | Assessment |
|-----------|------------|
| **Relevance** | Retrieved documents match query topic via semantic similarity |
| **Factual grounding** | Answers are generated from retrieved context only |
| **Completeness** | Top-5 retrieval captures diverse viewpoints on the topic |
| **Limitations** | Small corpus limits retrieval quality; GPT-2 may produce incoherent text |
| **Mitigation** | Source documents displayed alongside answers for human verification |

## B4. Ethics, Risk, and Responsible AI

This section provides a critical discussion of ethical considerations across both components.

In [ ]:
from component_b import ethics_discussion
ethics_discussion()

### Summary of Ethical Considerations

#### 1. Bias in Datasets
- **Accident data:** Under-reporting in rural/informal areas creates geographic bias. Models trained on this data will under-allocate resources to already-underserved communities.
- **Hansard transcripts:** Reflect only parliamentary voices, not public sentiment. English-language bias excludes contributions in isiZulu, Afrikaans, and other official languages.

#### 2. Transparency and Explainability
- Ensemble models offer feature importance but lack local explanations for individual predictions.
- LLMs are fundamentally opaque — attention weights are not reliable explanations.
- RAG improves transparency by providing source documents alongside generated answers.

#### 3. Risks of Harmful Content
- False confidence in "low risk" accident predictions could reduce safety vigilance.
- LLM-generated policy summaries may misrepresent political positions.
- Generated content could be used for political misinformation.

#### 4. Mitigation Strategies
- **Prompt engineering:** Constrain outputs to factual, cited summaries.
- **Content filtering:** Toxicity and factuality checks on all generated text.
- **Human-in-the-loop:** All recommendations require human review before action.
- **Confidence thresholds:** Only present high-confidence predictions.
- **Regular auditing:** Periodic bias audits across demographics and geography.

#### 5. South African Context
- **POPIA compliance:** Automated decisions must be transparent and contestable.
- **Linguistic inclusion:** 11 official languages require multilingual model support.
- **Historical equity:** Models must not perpetuate apartheid-era resource allocation patterns.
- **NDP 2030 alignment:** Technology must demonstrably serve all communities equitably.

---
## Conclusion

This project demonstrated:

1. **Component A:** A hybrid system combining ensemble learning (Random Forest, XGBoost) for accident severity prediction with Q-learning for adaptive intervention policy learning. The integrated system maps predicted risk levels to optimal safety interventions.

2. **Component B:** An NLP pipeline using transformer-based embeddings for sentiment classification of parliamentary transcripts, combined with a RAG system for policy-focused question answering.

3. **Ethics:** Both systems require careful consideration of bias, transparency, and responsible deployment within the South African context.

**Key recommendations:**
- Deploy as decision-support tools with mandatory human oversight
- Invest in multilingual data collection to reduce language bias
- Conduct regular bias audits before and after deployment
- Ensure POPIA compliance in all automated decision processes